## Colab session configuration

Connect to google cloud runtime and use GPU.

## Reading hg38 genome assembly 

Set parent directory.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import sys
import pathlib
from pathlib import Path
!pip install pyfaidx
from pyfaidx import Fasta

# Function to get the root directory
def get_root():
    """
    Get the root directory to access all files using Colab or local.

    Note that this function assumes the dir containing notebook is 1 below repo main dir.
    """
    COLAB = os.path.abspath(".") == "/content"

    if COLAB:
        root = "/content"
    else:
        root = Path.cwd().parent

    return root

# Set root
root = get_root()

# Set 
if root not in sys.path:
    sys.path.insert(0, str(root))

Download repo and the hg38 fasta zipped file.

Figure out a good way to run this pipeline only if colab is detected, then otherwise, just use local data folder.

In [ ]:
# Download repo to access scripts
!git clone https://github.com/eddykang06/mini-gLM.git
&cd mini-gLM

# Add repo scripts to path
sys.path.append("/content/mini-gLM")

# Download zipped human genome files
!mkdir -p data
!curl -L -C - "https://hgdownload.cse.ucsc.edu/goldenPath/hg38/bigZips/hg38.fa.gz" -o data/hg38.fa.gz
!curl -L -C - "https://hgdownload.cse.ucsc.edu/goldenPath/hg38/bigZips/hg38.chrom.sizes" -o data/hg38.chrom.sizes
!gunzip -c data/hg38.fa.gz > data/hg38.fa # Unzip to fasta

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  938M  100  938M    0     0  63.9M      0  0:00:14  0:00:14 --:--:-- 60.9M
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 11672  100 11672    0     0  93291      0 --:--:-- --:--:-- --:--:-- 94129


Sample random sections of the genome, weighted by chromosome length and within the specified sequence length range.

In [ ]:
from src import sample_from_fasta

# Set seed
rng = np.random.default_rng(111)

# Data directory
data_dir = root / "data"
fasta_file = data_dir / "hg38.fa"
chrom_size_file = data_dir / "hg38.chrom.sizes"

# Set parameters
n_seqs = 100
min_length = 20
max_length = 500

# Get pretraining data
pretraining = sample_from_fasta(
    fasta_file = fasta_file,
    chrom_size_file = chrom_size_file,
    n_seqs = n_seqs,
    min_length = min_length,
    max_length = max_length
)

ModuleNotFoundError: No module named 'src'

## Tokenizer

Simple function to generate random samples to mimic fasta samples.

In [ ]:
def generate_seqs(
        min_length,
        max_length,
        num_seqs
):
    seqs = []

    for i in range(num_seqs):
        length = np.random.choice(np.arange(min_length, max_length))
        seq = [np.random.choice(["A", "C", "G", "T"]) for i in range(length)]
        seq = "".join(seq)
        seqs.append(seq)

    return seqs

Do we need to map the tokens to integer IDs?? What would the final output of the tokenizer be for a new set of sequences??

Function to tokenize a set of sequence given a specified vocabulary and a set of merge rules.

In [ ]:
def tokenize_sequences(
    sequence_list : list[str],
    vocab: list,
    merge_rules: dict[tuple, str]
):  
    tokenized = []
    for seq in sequence_list:
        
        # Separate sequence into individual strings
        seq = list(seq)

        # Iteratively apply merge rules



        # Map final output to tokens IDs?

    return tokenized
    


    

Implementation of BPE tokenizer class.

In [ ]:
from src.tokenize import train_bpe_tokenizer


class BPE_Tokenizer():
    def __init__(
            self, 
            vocab: list[str], 
            merge_rules: dict[tuple: str], 
            token_to_idx: dict[str: int],
            idx_to_token: dict[int: str]
        ):
        self.vocab = vocab
        self.vocab_size = len(self.vocab)
        self.merge_rules = merge_rules
        self.token_to_idx = token_to_idx
        self.idx_to_token = idx_to_token
    
    # Train a tokenizer on a given corpus of DNA sequences
    def train(self, sequence_list, final_vocab_size, seed):

        # Train tokenizer on list of sequences
        vocab, merge_rules = train_bpe_tokenizer(
            sequence_list = sequence_list,
            final_vocab_size = final_vocab_size,
            seed = seed
        )

        # Create vector of token IDs
        idx = list(range(len(vocab)))

        # Update vocab, merge rules, and token ID mapping
        self.vocab = vocab
        self.merge_rules = merge_rules
        self.token_to_idx = {token: id for token, id in zip(vocab, idx)}
        self.idx_to_token = {id: token for id, token in zip(idx, vocab)}
    
    # Tokenize a given sequence
    def tokenize(self, sequence_list):

        tokenized = tokenize_sequences(
            sequence_list = sequence_list,
            vocab = self.vocab,
            merge_rules = self.merge_rules
        ) 

        return tokenized

    # Train a tokenizer and transform the corpus as well
    def train_tokenize(self, sequence_list, final_vocab_size, seed):

        return 
    
    # Map a list of tokens to token IDs
    def encode(self):

        return 

    # Map a list of token IDs to tokens
    def decode(self):

        return
    
    # Clear trained params?
    def clear(self):






Reference: https://huggingface.co/learn/llm-course/en/chapter6/5

## Model architecture

In [ ]:
!pip install accelerate
!pip install datasets
!pip install transformers
!pip install torch
!pip install flash-attn

import accelerate
import flash_attn
import torch
import torch.nn as nn
import torch.nn.functional as F
import transformers
from datasets import load_dataset
from transformers import (
    AutoConfig,
    AutoModelForCausalLM,
    AutoTokenizer,
    DataCollatorForLanguageModeling,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
)

torch.backends.cudnn.benchmark=True

Implement a sparse attention transformer.

Implement the model with 10 transformer blocks.

In [ ]:
class miniglm(nn.Module):

    def __init__(self):